In [ ]:
!pip -q install ultralytics opencv-python pandas numpy

import cv2
import numpy as np
import pandas as pd
import os
from ultralytics import YOLO
from collections import defaultdict

# ==============================
mkvideo="inputvideo1.mp4"
# CONFIG
# ==============================
VIDEO_PATH = mkvideo   # <-- change to your video path
OUT_VIDEO = "output_abnormal_detection.mp4"
EVIDENCE_DIR = "evidence"
CSV_PATH = "abnormal_report.csv"

CONF_THRES = 0.3
ABNORMAL_THRES = 0.6
MAX_TRAJ = 30   # keep last 30 points

os.makedirs(EVIDENCE_DIR, exist_ok=True)

# ==============================
# LOAD MODEL
# ==============================
model = YOLO("yolov8n.pt")  # for speed; can use yolov8s.pt for better accuracy

# ==============================
# VIDEO SETUP
# ==============================
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "❌ Error: Could not open video. Check VIDEO_PATH"

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (w, h))

print("✅ Video loaded")
print("FPS:", fps, "Resolution:", (w, h))

# ==============================
# DATA STRUCTURES
# ==============================
trajectories = defaultdict(list)
report_rows = []

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    # YOLO tracking (ByteTrack)
    results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=CONF_THRES, verbose=False)

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes
        ids = boxes.id.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy()

        for vid, box in zip(ids, xyxy):
            x1, y1, x2, y2 = box.astype(int)
            cx, cy = (x1 + x2)//2, (y1 + y2)//2

            # save trajectory point
            trajectories[vid].append((cx, cy))
            if len(trajectories[vid]) > MAX_TRAJ:
                trajectories[vid] = trajectories[vid][-MAX_TRAJ:]

            # (temporary) draw only tracking for now
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, f"ID:{vid}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            # draw trajectory line
            pts = trajectories[vid]
            for i in range(1, len(pts)):
                cv2.line(frame, pts[i-1], pts[i], (255, 255, 0), 2)

    writer.write(frame)

cap.release()
writer.release()

print("✅ Output saved:", OUT_VIDEO)
print("✅ Evidence folder created:", EVIDENCE_DIR)


In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from ultralytics import YOLO
from collections import defaultdict

# ==============================
# CONFIG
# ==============================
VIDEO_PATH = mkvideo
OUT_VIDEO = "output_abnormal_detection.mp4"
EVIDENCE_DIR = "evidence"
CSV_PATH = "abnormal_report.csv"

CONF_THRES = 0.3

# abnormality scoring
ABNORMAL_THRES = 0.6
ALPHA = 0.4
BETA  = 0.3
GAMMA = 0.3

MAX_TRAJ = 30
MIN_POINTS_FOR_DECISION = 10
SMOOTH_WINDOW = 5

# Only vehicles
VEHICLE_CLASSES = {2, 3, 5, 7}   # car, motorcycle, bus, truck

os.makedirs(EVIDENCE_DIR, exist_ok=True)

# ==============================
# LOAD MODEL
# ==============================
model = YOLO("yolov8n.pt")

# ==============================
# VIDEO SETUP
# ==============================
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "❌ Error: Could not open video. Check VIDEO_PATH"

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (w, h))

print("✅ Video loaded")
print("FPS:", fps, "Resolution:", (w, h))

# ==============================
# HELPERS
# ==============================
def safe_norm(x):
    # stabilizes score into 0–1 range
    return float(np.clip(np.tanh(x), 0, 1))

def compute_speed(p1, p2, fps):
    dist = np.linalg.norm(np.array(p2) - np.array(p1))
    return dist * fps

def direction_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return np.degrees(np.arctan2(dy, dx))

# ==============================
# DATA STRUCTURES
# ==============================
trajectories = defaultdict(list)
speed_hist = defaultdict(list)
score_hist = defaultdict(list)

prev_abnormal_state = defaultdict(lambda: False)  # for event detection
report_rows = []

frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    timestamp = frame_id / fps

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=CONF_THRES,
        verbose=False
    )

    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes
        ids = boxes.id.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy()
        cls_list = boxes.cls.cpu().numpy().astype(int)

        for i, (vid, box) in enumerate(zip(ids, xyxy)):
            cls = cls_list[i]

            # ✅ Track only vehicles
            if cls not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = box.astype(int)
            cx, cy = (x1 + x2)//2, (y1 + y2)//2

            trajectories[vid].append((cx, cy))
            if len(trajectories[vid]) > MAX_TRAJ:
                trajectories[vid] = trajectories[vid][-MAX_TRAJ:]

            pts = trajectories[vid]

            # default scores
            abnormal_score = 0.0
            speed_var_score = 0.0
            dir_change_score = 0.0
            jerk_score = 0.0

            # speed history
            if len(pts) >= 2:
                s = compute_speed(pts[-2], pts[-1], fps)
                speed_hist[vid].append(s)
                if len(speed_hist[vid]) > MAX_TRAJ:
                    speed_hist[vid] = speed_hist[vid][-MAX_TRAJ:]

            # ---------- compute scores ----------
            if len(speed_hist[vid]) >= 5:
                speed_std = np.std(speed_hist[vid][-10:])
                speed_var_score = safe_norm(speed_std / 50.0)

            if len(pts) >= 4:
                a1 = direction_angle(pts[-4], pts[-3])
                a2 = direction_angle(pts[-2], pts[-1])
                da = abs(a2 - a1)
                da = min(da, 360 - da)
                dir_change_score = safe_norm(da / 60.0)

                x1_, x2_, x3_, x4_ = pts[-4][0], pts[-3][0], pts[-2][0], pts[-1][0]
                v1 = (x2_ - x1_) * fps
                v2 = (x3_ - x2_) * fps
                v3 = (x4_ - x3_) * fps
                a1_ = (v2 - v1) * fps
                a2_ = (v3 - v2) * fps
                jerk = abs(a2_ - a1_)
                jerk_score = safe_norm(jerk / 5000.0)

            abnormal_score = (ALPHA * speed_var_score +
                              BETA  * dir_change_score +
                              GAMMA * jerk_score)

            # ✅ Smooth score to reduce flicker
            score_hist[vid].append(abnormal_score)
            if len(score_hist[vid]) > MAX_TRAJ:
                score_hist[vid] = score_hist[vid][-MAX_TRAJ:]

            smooth_score = float(np.mean(score_hist[vid][-SMOOTH_WINDOW:]))

            # ✅ Warm-up protection
            if len(pts) < MIN_POINTS_FOR_DECISION:
                is_abnormal = False
            else:
                is_abnormal = smooth_score > ABNORMAL_THRES

            # ---------- visualization ----------
            if is_abnormal:
                color = (0, 0, 255)
                label = f"ABNORMAL {smooth_score:.2f}"
            else:
                color = (0, 255, 0)
                label = f"Normal {smooth_score:.2f}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID:{vid} {label}", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            # trajectory
            for j in range(1, len(pts)):
                cv2.line(frame, pts[j-1], pts[j], (255, 255, 0), 2)

            # ---------- event-based evidence saving ----------
            if is_abnormal and not prev_abnormal_state[vid]:
                evidence_path = os.path.join(EVIDENCE_DIR, f"vehicle_{vid}_event_frame_{frame_id}.jpg")
                cv2.imwrite(evidence_path, frame)

                report_rows.append({
                    "vehicle_id": vid,
                    "frame_id": frame_id,
                    "timestamp_sec": round(timestamp, 2),
                    "smooth_abnormal_score": round(smooth_score, 3),
                    "speed_var_score": round(speed_var_score, 3),
                    "dir_change_score": round(dir_change_score, 3),
                    "jerk_score": round(jerk_score, 3),
                    "evidence_path": evidence_path
                })

            prev_abnormal_state[vid] = is_abnormal

    writer.write(frame)

cap.release()
writer.release()

df = pd.DataFrame(report_rows)
df.to_csv(CSV_PATH, index=False)

print("✅ Output saved:", OUT_VIDEO)
print("✅ Evidence saved in:", EVIDENCE_DIR)
print("✅ CSV report saved:", CSV_PATH)
print("✅ Total abnormal events:", len(df))

df.head()


In [ ]:
import pandas as pd

df = pd.read_csv("abnormal_report.csv")
print("✅ Total events:", len(df))

# Per-vehicle summary
summary = df.groupby("vehicle_id").agg(
    abnormal_events=("vehicle_id", "count"),
    avg_score=("smooth_abnormal_score", "mean"),
    max_score=("smooth_abnormal_score", "max"),
    first_time=("timestamp_sec", "min"),
    last_time=("timestamp_sec", "max")
).reset_index()

summary = summary.sort_values(by="abnormal_events", ascending=False)

print("✅ Vehicles flagged:", summary.shape[0])
summary
summary.to_csv("vehicle_abnormal_summary.csv", index=False)
print("✅ Saved: vehicle_abnormal_summary.csv")


In [ ]:
import matplotlib.pyplot as plt

# bar plot: abnormal events per vehicle
plt.figure(figsize=(10,5))
plt.bar(summary["vehicle_id"].astype(str), summary["abnormal_events"])
plt.xticks(rotation=45)
plt.title("Abnormal Events per Vehicle ID")
plt.xlabel("Vehicle ID")
plt.ylabel("Count of Abnormal Events")
plt.show()

# timeline plot for top 1 vehicle
top_vid = summary.iloc[0]["vehicle_id"]
df_top = df[df["vehicle_id"] == top_vid].sort_values("timestamp_sec")

plt.figure(figsize=(10,5))
plt.plot(df_top["timestamp_sec"], df_top["smooth_abnormal_score"], marker="o")
plt.title(f"Abnormal Score Timeline for Vehicle {top_vid}")
plt.xlabel("Timestamp (sec)")
plt.ylabel("Smooth Abnormal Score")
plt.show()
